# Full Machine Learning Pipeline

## Job 1: ETL to Build the Warehouse
---

In [ ]:
import sqlite3
import pandas as pd
import os


In [1]:
src = sqlite3.connect('shop.db')

orders      = pd.read_sql('SELECT * FROM orders', src)
customers   = pd.read_sql('SELECT * FROM customers', src)
shipments   = pd.read_sql('SELECT * FROM shipments', src)
order_items = pd.read_sql('''
    SELECT oi.order_id,
           oi.quantity, oi.unit_price, oi.line_total,
           p.category
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
''', src)
reviews = pd.read_sql('SELECT customer_id, rating FROM product_reviews', src)

src.close()


NameError: name 'sqlite3' is not defined

In [ ]:
items_agg = order_items.groupby('order_id').agg(
    items_count         = ('quantity', 'count'),
    total_quantity      = ('quantity', 'sum'),
    avg_unit_price      = ('unit_price', 'mean'),
    max_unit_price      = ('unit_price', 'max'),
    min_unit_price      = ('unit_price', 'min'),
    distinct_categories = ('category', 'nunique'),
).reset_index()

cat_dummies = (
    order_items.drop_duplicates(subset=['order_id', 'category'])
    .assign(present=1)
    .pivot_table(index='order_id', columns='category', values='present', fill_value=0)
    .reset_index()
)
cat_dummies.columns = ['order_id'] + [f'cat_{c.lower()}' for c in cat_dummies.columns[1:]]

items_agg = items_agg.merge(cat_dummies, on='order_id', how='left')


In [ ]:
reviews_agg = reviews.groupby('customer_id').agg(
    customer_avg_rating   = ('rating', 'mean'),
    customer_review_count = ('rating', 'count'),
).reset_index()


In [ ]:
df = orders.copy()

cust_cols = ['customer_id', 'gender', 'birthdate', 'created_at',
             'city', 'state', 'zip_code', 'customer_segment',
             'loyalty_tier', 'is_active']
df = df.merge(customers[cust_cols], on='customer_id', how='left')

ship_cols = [c for c in shipments.columns if c != 'shipment_id']
df = df.merge(shipments[ship_cols], on='order_id', how='left')

df = df.merge(items_agg, on='order_id', how='left')
df = df.merge(reviews_agg, on='customer_id', how='left')

print(df.shape)
df.head()


In [ ]:
if os.path.exists('warehouse.db'):
    os.remove('warehouse.db')

dst = sqlite3.connect('warehouse.db')
df.to_sql('orders_denorm', dst, index=False, if_exists='replace')
dst.close()

print('warehouse.db written — table: orders_denorm')


## Job 2: Train the Model and Save the Artifacts
---

In [ ]:
# jobs/train_model.py
import json
from datetime import datetime
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression

from config import WH_DB_PATH, ARTIFACTS_DIR, MODEL_PATH, MODEL_METADATA_PATH, METRICS_PATH
from utils_db import sqlite_conn

MODEL_VERSION = "1.0.0"

In [ ]:
def train_and_save():
	with sqlite_conn(WH_DB_PATH) as conn:
		df = pd.read_sql("SELECT * FROM modeling_orders", conn)
	
	label_col = "risk_score" # Label column - predicting the risk score
	
	feature_cols = [c for c in df.columns if c != label_col]
	X = df[feature_cols]
	y = df[label_col].astype(int)
	
	X_train, X_test, y_train, y_test = train_test_split(
		X, y, test_size=0.2, random_state=42, stratify=y
	)
	
	numeric_features = ["num_items", "total_value", "avg_weight", "customer_age", "order_dow", "order_month"]
	categorical_features = []
	
	numeric_pipe = Pipeline(steps=[
		("imputer", SimpleImputer(strategy="median"))
	])
	
	categorical_pipe = Pipeline(steps=[
		("imputer", SimpleImputer(strategy="most_frequent")),
		("onehot", OneHotEncoder(handle_unknown="ignore"))
	])
	
	preprocessor = ColumnTransformer(
		transformers=[
			("num", numeric_pipe, numeric_features),
			("cat", categorical_pipe, categorical_features)
		],
		remainder="drop"
	)
	
	clf = LogisticRegression(max_iter=500)
	
	model = Pipeline(steps=[
		("prep", preprocessor),
		("clf", clf)
	])
	
	model.fit(X_train, y_train)
	
	y_pred = model.predict(X_test)
	y_prob = model.predict_proba(X_test)[:, 1]
	
	metrics = {
		"accuracy": float(accuracy_score(y_test, y_pred)),
		"f1": float(f1_score(y_test, y_pred)),
		"roc_auc": float(roc_auc_score(y_test, y_prob)),
		"row_count_train": int(len(X_train)),
		"row_count_test": int(len(X_test))
	}
	
	ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
	
	joblib.dump(model, str(MODEL_PATH))
	
	metadata = {
		"model_version": MODEL_VERSION,
		"trained_at_utc": datetime.utcnow().isoformat(),
		"feature_list": feature_cols,
		"label": label_col,
		"warehouse_table": "modeling_orders",
		"warehouse_rows": int(len(df))
	}
	
	with open(MODEL_METADATA_PATH, "w", encoding="utf-8") as f:
		json.dump(metadata, f, indent=2)
	
	with open(METRICS_PATH, "w", encoding="utf-8") as f:
		json.dump(metrics, f, indent=2)
	
	print("Training complete.")
	print(f"Saved model: {MODEL_PATH}")
	print(f"Saved metadata: {MODEL_METADATA_PATH}")
	print(f"Saved metrics: {METRICS_PATH}")

if __name__ == "__main__":
    train_and_save()

## Job 3: Run Inference and Write Predictions to `shop.db`
---